In [1]:
%pip install -U ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [1]:
print("Customer Churn Intelligence")
print("Python environment is working!")

Customer Churn Intelligence
Python environment is working!


In [2]:
import pandas as pd

print("Pandas version:", pd.__version__)

Pandas version: 3.0.5


In [3]:
import duckdb

print("DuckDB is working!")


DuckDB is working!


In [4]:
from pathlib import Path

data_path = Path("../data/raw/hm")

print("Data folder:", data_path)
print("Files found:")

for file in data_path.iterdir():
    print(file.name)

Data folder: ..\data\raw\hm
Files found:
.gitkeep
articles.csv
customers.csv
transactions_train.csv


In [5]:
import duckdb

con = duckdb.connect()

con.sql("""
SELECT COUNT(*) AS total_transactions
FROM read_csv_auto('../data/raw/hm/transactions_train.csv')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│ total_transactions │
│       int64        │
├────────────────────┤
│           31788324 │
└────────────────────┘



In [6]:
import duckdb

con = duckdb.connect()

In [ ]:
con.sql("""
SELECT COUNT(DISTINCT customer_id) AS distinct_customers
FROM read_csv_auto('../data/raw/hm/transactions_train.csv')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│ distinct_customers │
│       int64        │
├────────────────────┤
│            1362281 │
└────────────────────┘



In [9]:
con.sql("""
SELECT
    MIN(t_dat) AS earliest_date,
    MAX(t_dat) AS latest_date
FROM read_csv_auto('../data/raw/hm/transactions_train.csv')
""").show()

┌───────────────┬─────────────┐
│ earliest_date │ latest_date │
│     date      │    date     │
├───────────────┼─────────────┤
│ 2018-09-20    │ 2020-09-22  │
└───────────────┴─────────────┘



In [10]:
con.sql("""
SELECT COUNT(*) AS total_customers
FROM read_csv_auto('../data/raw/hm/customers.csv')
""").show()

┌─────────────────┐
│ total_customers │
│      int64      │
├─────────────────┤
│         1371980 │
└─────────────────┘



In [11]:
con.sql("""
DESCRIBE
SELECT *
FROM read_csv_auto('../data/raw/hm/customers.csv')
""").show()

┌────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │ column_type │  null   │   key   │ default │  extra  │
│        varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ FN                     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Active                 │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ club_member_status     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ fashion_news_frequency │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ age                    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ postal_code            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [12]:
con.sql("""
SELECT
    COUNT(*) AS total_customers,
    COUNT(*) FILTER (WHERE age IS NULL) AS missing_age,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE age IS NULL) / COUNT(*),
        2
    ) AS missing_age_percentage
FROM read_csv_auto('../data/raw/hm/customers.csv')
""").show()

┌─────────────────┬─────────────┬────────────────────────┐
│ total_customers │ missing_age │ missing_age_percentage │
│      int64      │    int64    │         double         │
├─────────────────┼─────────────┼────────────────────────┤
│         1371980 │       15861 │                   1.16 │
└─────────────────┴─────────────┴────────────────────────┘



In [13]:
con.sql("""
SELECT
    customer_id,
    t_dat,
    LAG(t_dat) OVER (
        PARTITION BY customer_id
        ORDER BY t_dat
    ) AS previous_purchase
FROM read_csv_auto('../data/raw/hm/transactions_train.csv')
LIMIT 20
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────────────────────────────────┬────────────┬───────────────────┐
│                           customer_id                            │   t_dat    │ previous_purchase │
│                             varchar                              │    date    │       date        │
├──────────────────────────────────────────────────────────────────┼────────────┼───────────────────┤
│ 185ad298ca5b00051189ccbe1c65a73bc861492f7347b00d4a041c7034b38a72 │ 2020-03-27 │ 2020-03-27        │
│ 185ad298ca5b00051189ccbe1c65a73bc861492f7347b00d4a041c7034b38a72 │ 2020-03-27 │ 2020-03-27        │
│ 185ad298ca5b00051189ccbe1c65a73bc861492f7347b00d4a041c7034b38a72 │ 2020-03-27 │ 2020-03-27        │
│ 185ad298ca5b00051189ccbe1c65a73bc861492f7347b00d4a041c7034b38a72 │ 2020-03-27 │ 2020-03-27        │
│ 185ad298ca5b00051189ccbe1c65a73bc861492f7347b00d4a041c7034b38a72 │ 2020-03-27 │ 2020-03-27        │
│ 185ad298ca5b00051189ccbe1c65a73bc861492f7347b00d4a041c7034b38a72 │ 2020-03-27 │ 

In [14]:
con.sql("""
WITH unique_purchases AS (
    SELECT DISTINCT
        customer_id,
        t_dat
    FROM read_csv_auto('../data/raw/hm/transactions_train.csv')
),

purchases_with_previous AS (
    SELECT
        customer_id,
        t_dat,
        LAG(t_dat) OVER (
            PARTITION BY customer_id
            ORDER BY t_dat
        ) AS previous_purchase
    FROM unique_purchases
),

gaps AS (
    SELECT
        customer_id,
        date_diff('day', previous_purchase, t_dat) AS gap_days
    FROM purchases_with_previous
    WHERE previous_purchase IS NOT NULL
)

SELECT
    MIN(gap_days) AS minimum_gap,
    quantile_cont(gap_days, 0.25) AS percentile_25,
    quantile_cont(gap_days, 0.50) AS median_gap,
    quantile_cont(gap_days, 0.75) AS percentile_75,
    quantile_cont(gap_days, 0.90) AS percentile_90,
    MAX(gap_days) AS maximum_gap
FROM gaps
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬───────────────┬────────────┬───────────────┬───────────────┬─────────────┐
│ minimum_gap │ percentile_25 │ median_gap │ percentile_75 │ percentile_90 │ maximum_gap │
│    int64    │    double     │   double   │    double     │    double     │    int64    │
├─────────────┼───────────────┼────────────┼───────────────┼───────────────┼─────────────┤
│           1 │           7.0 │       22.0 │          58.0 │         124.0 │         731 │
└─────────────┴───────────────┴────────────┴───────────────┴───────────────┴─────────────┘

